# Mini Project 3: Chat with Your Notes

This notebook demonstrates a complete PDF question-answering workflow: PDF extraction, page-aware chunks, embeddings, cosine-similarity retrieval, source references, an extractive grounded answer, and an abstention path. A Streamlit UI is provided in `app.py` for interactive questions.

Default mode does not require a paid API or local generative model. Optional Ollama generation is included in the Streamlit app.

In [1]:
from pathlib import Path
import sys, pandas as pd
HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "rag_core.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the Week 3 project root in VS Code.")
sys.path.insert(0, str(PROJECT_ROOT))
from rag_core import extract_pdf_pages, chunk_pages, SemanticRetriever, extractive_answer, NO_ANSWER
MINI_DIR = PROJECT_ROOT / "Mini_Project_3_Chat_with_Your_Notes"
pages = extract_pdf_pages(MINI_DIR / "sample_study_notes.pdf")
chunks = chunk_pages(pages, chunk_size=120, overlap=25)
retriever = SemanticRetriever(backend="auto").fit(chunks)
print(f"Pages={len(pages)}, chunks={len(chunks)}, backend={retriever.backend}")
if retriever.warning: print(retriever.warning)

Pages=8, chunks=15, backend=tfidf
Dense embedding model unavailable; using TF-IDF fallback. Details: ModuleNotFoundError: No module named 'sentence_transformers'


## Ask a question

Edit the question below and rerun the cell. The answer uses sentences from the retrieved source passages; page numbers are retained for traceability.

In [2]:
question = "How is precision calculated and what does it mean?"  # Try another question here
hits = retriever.search(question, top_k=4)
answer, answer_pages = extractive_answer(question, hits)
if not hits or hits[0]["score"] < (0.20 if retriever.backend == "tfidf" else 0.30):
    answer, answer_pages = NO_ANSWER, []
print("QUESTION:", question)
print("ANSWER:", answer)
print("SOURCE PDF PAGES:", answer_pages)
print("\nRETRIEVED PASSAGES:")
for h in hits:
    c = h["chunk"]
    print(f"\n[p. {c.page} | similarity={h['score']:.3f}]\n{c.text}")

QUESTION: How is precision calculated and what does it mean?
ANSWER: I could not find this in the document
SOURCE PDF PAGES: []

RETRIEVED PASSAGES:

[p. 6 | similarity=0.178]
truly positive? Recall Recall is true positives divided by actual positives. It answers: among all actual positive items, what fraction did the model find? F1 score The F1 score is the harmonic mean of precision and recall. It is useful when both false positives and false negatives matter. Metric choice should reflect the practical cost of different errors.

[p. 6 | similarity=0.116]
SkillNexis LLM Development - Week 3 sample notes PDF page 6 Model Evaluation Metrics Confusion matrix For binary classification, a confusion matrix counts true positives, true negatives, false positives, and false negatives. These counts help explain the kinds of mistakes a classifier makes. Accuracy Accuracy is the fraction of predictions that are correct: (true positives + true negatives) divided by all predictions. Accuracy can be

## Run the ten-question test set

This creates a local results file you can attach to your submission. Check each answer and page citation manually. The two unsupported questions should abstain when the threshold is appropriate; similarity thresholds are heuristics, not formal proof of evidence.

In [3]:
test_df = pd.read_csv(MINI_DIR / "test_questions.csv")
rows=[]
for _, q in test_df.iterrows():
    hits = retriever.search(q["question"], top_k=4)
    if not hits or hits[0]["score"] < (0.20 if retriever.backend == "tfidf" else 0.30):
        answer, source_pages = NO_ANSWER, []
    else:
        answer, source_pages = extractive_answer(q["question"], hits)
    rows.append({"question_id":q["question_id"],"question":q["question"],"answer":answer,"source_pdf_pages":";".join(map(str,source_pages)),"top1_similarity":round(hits[0]["score"],4) if hits else 0.0})
mini_results = pd.DataFrame(rows)
mini_results.to_csv(MINI_DIR / "generated_test_answers.csv", index=False, encoding="utf-8")
display(mini_results)
print("Saved:", MINI_DIR / "generated_test_answers.csv")

,question_id,question,answer,source_pdf_pages,top1_similarity
0,Q01,What is machine learning?,What is machine learning? SkillNexis LLM Devel...,1;2,0.2771
1,Q02,How does supervised learning learn from examples?,SkillNexis LLM Development - Week 3 sample not...,1;2;3,0.2143
2,Q03,What is the difference between classification ...,I could not find this in the document,,0.1325
3,Q04,What does clustering do in unsupervised learning?,SkillNexis LLM Development - Week 3 sample not...,1;4,0.2120
4,Q05,"What is overfitting, and name two ways to redu...",I could not find this in the document,,0.1727
5,Q06,"What are the roles of the training, validation...",SkillNexis LLM Development - Week 3 sample not...,5,0.2434
6,Q07,How is precision calculated and what does it m...,I could not find this in the document,,0.1782
7,Q08,"What is a text embedding, and what does cosine...",A text embedding represents text as a numeric ...,7,0.2721
8,Q09,What exact accuracy percentage did the example...,I could not find this in the document,,0.1605
9,Q10,In what year was the first GPT model released?,I could not find this in the document,,0.0909


Saved: /mnt/data/SkillNexis_LLM_Development_Week_3/Mini_Project_3_Chat_with_Your_Notes/generated_test_answers.csv


## Manual checks

1. Ask an answerable question, such as **What is precision?**. Confirm the answer matches the source passage and shows its PDF page.
2. Ask **What exact accuracy percentage did the example model achieve?**. The sample notes contain a definition of accuracy but no model-specific percentage; this is an abstention test.
3. Ask **In what year was the first GPT model released?**. This detail is intentionally absent.
4. Inspect retrieved passages even when the app abstains. Do not treat vector similarity alone as proof that evidence supports a claim.